In [1]:
import os
import cv2
import numpy as np
import pandas as pd

# ============================================================
# Select Dataset for train and augmented change path manually
# ============================================================

dataset = input("Which dataset do you want to label? (train / valid / test / train_2): ").strip().lower()

BASE_FOLDER = r"../data_set"

if dataset == "train":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train")
    CSV_FILE = os.path.join(BASE_FOLDER, "train.csv")

elif dataset in ["valid", "validation", "val"]:
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "valid")
    CSV_FILE = os.path.join(BASE_FOLDER, "valid.csv")

elif dataset == "test":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "test")
    CSV_FILE = os.path.join(BASE_FOLDER, "test.csv")
    
elif dataset == "train_2":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train_2")
    CSV_FILE = os.path.join(BASE_FOLDER, "train_2.csv")

else:
    print("Invalid dataset.")
    exit()

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# ============================================================
# Helper function to get base name
# ============================================================
def get_original_stem(filename):
    """ Extracts base file string before '_crop_X' or '_no_detection' """
    stem = os.path.splitext(filename)[0]
    if "_no_detection" in stem:
        return stem.split("_no_detection")[0]
    if "_crop_" in stem:
        return stem.split("_crop_")[0]
    return stem

# ============================================================
# Resume Support & Stem Tracking
# ============================================================

if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE, dtype={"label": str})
    labeled_images = set(df["image"].astype(str))
    # Pre-calculate stems of already completed images for fast, exact matching
    labeled_stems = {get_original_stem(img) for img in labeled_images}
else:
    df = pd.DataFrame(columns=["image", "label"])
    labeled_images = set()
    labeled_stems = set()

# ============================================================
# Image List
# ============================================================

images = sorted([
    img for img in os.listdir(IMAGE_FOLDER)
    if img.lower().endswith(SUPPORTED_EXTENSIONS)
])

print(f"\nDataset : {dataset}")
print(f"Total Images found in folder : {len(images)}")
print(f"Already Labeled in CSV       : {len(labeled_images)}")

# ============================================================
# Labeling Loop
# ============================================================

for index, image_name in enumerate(images):

    # FIX: Exact set matching prevents substring partial match bugs
    base_stem = get_original_stem(image_name)
    if image_name in labeled_images or base_stem in labeled_stems:
        continue

    image_path = os.path.join(IMAGE_FOLDER, image_name)
    original_img = cv2.imread(image_path)

    if original_img is None:
        print(f"Could not read {image_name}")
        continue

    # Keep a running processed version of the image
    processed_img = original_img.copy()

    # Helper function to refresh the OpenCV display window
    def update_display(img_to_show):
        h, w = img_to_show.shape[:2]
        scale = min(900 / w, 400 / h)
        display = cv2.resize(img_to_show, (int(w * scale), int(h * scale)))
        cv2.namedWindow("Meter Labeling Tool", cv2.WINDOW_NORMAL)
        cv2.imshow("Meter Labeling Tool", display)
        cv2.resizeWindow("Meter Labeling Tool", 900, 400)
        cv2.waitKey(100)

    # Initial display render
    update_display(processed_img)

    print(f"\n[{index+1}/{len(images)}] {image_name}")
    
    while True:
        label = input("Enter label (q=quit, s=skip, d=delete, v=smooth, b=sharpen): ").strip()

        # -----------------------------
        # OPTION: Quit
        # -----------------------------
        if label.lower() == "q":
            cv2.destroyAllWindows()
            df.to_csv(CSV_FILE, index=False)
            print("\nProgress Saved.")
            exit()

        # -----------------------------
        # OPTION: Skip
        # -----------------------------
        if label.lower() == "s":
            break

        # -----------------------------
        # OPTION: Delete
        # -----------------------------
        if label.lower() in ["d", "del", "delete"]:
            cv2.destroyWindow("Meter Labeling Tool") # Close window to avoid OS file locks
            try:
                os.remove(image_path)
                print(f"🔥 Deleted file: {image_name}")
            except Exception as e:
                print(f"Error deleting file: {e}")
            break

        # -----------------------------
        # FILTER OPTION: Smooth (v)
        # -----------------------------
        elif label.lower() == "v":
            # Bilateral filter reduces noise while keeping edges sharp
            processed_img = cv2.bilateralFilter(processed_img, d=9, sigmaColor=75, sigmaSpace=75)
            update_display(processed_img)
            print("✨ Applied Smoothing (Bilateral Filter)")
            continue

        # -----------------------------
        # FILTER OPTION: Sharpen (b)
        # -----------------------------
        elif label.lower() == "b":
            # Sharpening kernel
            kernel = np.array([[0, -1, 0], 
                               [-1, 5, -1], 
                               [0, -1, 0]])
            processed_img = cv2.filter2D(processed_img, -1, kernel)
            update_display(processed_img)
            print("⚡ Applied Sharpening Filter")
            continue

        # -----------------------------
        # OPTION: Save 5-digit Label
        # -----------------------------
        if len(label) == 5 and label.isdigit():
            # If the image was filtered, we save the image name with the label 
            # and write the label into the CSV.
            df.loc[len(df)] = [image_name, label]
            df.to_csv(CSV_FILE, index=False)
            
            labeled_images.add(image_name)
            labeled_stems.add(base_stem)
            break

        print("Invalid label! Please enter exactly 5 digits or use a valid control command.")

cv2.destroyAllWindows()
print("\n===================================")
print("All pending images have been processed!")
print("CSV saved to:")
print(CSV_FILE)
print("===================================")


Dataset : valid
Total Images found in folder : 232
Already Labeled in CSV       : 200

[28/232] 04791520002_crop_0.png
🔥 Deleted file: 04791520002_crop_0.png

[29/232] 05012645015_crop_0.png

[30/232] 05018274661_crop_0.png

[31/232] 05164210006_crop_0.png

[32/232] 05170659089_crop_0.png

[33/232] 05363200006_crop_0.png

[34/232] 05397555698_crop_0.png

[35/232] 05405396762_crop_0.png

[36/232] 05777877613_crop_0.png
🔥 Deleted file: 05777877613_crop_0.png

[37/232] 05831104103_crop_0.png

[38/232] 05969900009_crop_0.png

[39/232] 06014514688_crop_0.png

[40/232] 06024230796_crop_0.png

[41/232] 06159593703_crop_0.png

[42/232] 06229057705_crop_0.png

[45/232] 0642_73697310008_crop_0.png

[50/232] 0642_76274002658_crop_0.png

[54/232] 0642_78668786359_crop_0.png
🔥 Deleted file: 0642_78668786359_crop_0.png

[63/232] 06478453266_crop_0.png

[64/232] 06694480184_crop_0.png

[65/232] 06998177171_crop_0.png

[66/232] 07143674674_crop_0.png

[67/232] 07239504629_crop_0.png
🔥 Deleted file: 0